Last Update = 06/2025
<h2> Instruction</h2> 

this code extracts Scrape data manually for each  II_USA_xxxx :

    - FEDPHIL, 
    - FEDBOS, 
    - FEDCLE, 
    - FEDATLANTA, 
    - FEDCHICAGO, 
    - FEDSTLOUIS, 
    - FEDMINNEAPOLIS
    - II_USA FEDSF
regulator from the ZIP file, and save each excel file in folder 'tempfolder' . the ZIP file and this code file must be in the same directory.

- Before executing this code, download the ZIP file **CSV_ATTRIBUTES_ACTIVE.zip** on https://www.ffiec.gov/npw/FinancialReport/DataDownload --> CSV Download --> ZIP Attributes-Active.
- Move the ZIP file to the same folder as the code file (**.py** / **.ipynb**).
- Finally, run the code.
- the output excel file save in the folder : **tempfolder**


In [ ]:
! pip install reportlab
export_pdf = False  # Is set to “False” by default. Set to “True” if you wish to generate PDFs for each list of each controller.

In [4]:
#------------------------------------------------ Begin_Librairie ----------------------------------------
import datetime
import pandas as pd
from pandas import ExcelWriter
from time import sleep
import os
from zipfile import ZipFile


from reportlab.lib.pagesizes import letter
from reportlab.pdfgen import canvas
#------------------------------------------------ Begin_ fileName ----------------------------------------

regulatorName = 'II_USA FEDPHIL'

regulators = {
'II_USA FEDPHIL' : {'II_USA FEDPHIL 1'  : ['SMB', 'State Member Bank'],
                    'II_USA FEDPHIL 2'  : ['AGI', 'Agreement Corporation - Investment'], 
                    'II_USA FEDPHIL 3'  : ['BHC', 'Bank Holding Company'], 
                    'II_USA FEDPHIL 4'  : ['EDI', 'Edge Corporation - Investment'], 
                    'II_USA FEDPHIL 5'  : ['FHD', 'Financial Holding Company / BHC (Note: Can be a domestic or foreign domiciled holding company)'],
                    'II_USA FEDPHIL 6'  : ['IHC', 'Intermediate Holding Company'],
                    'II_USA FEDPHIL 7'  : ['SLHC', 'Savings and Loan Holding Company']
                    },
'II_USA FEDATLANTA':{'II_USA FEDATLANTA 1' : ['SMB', 'State Member Bank'],
                    'II_USA FEDATLANTA 2'  : ['AGI', 'Agreement Corporation - Investment'], 
                    'II_USA FEDATLANTA 3'  : ['BHC', 'Bank Holding Company'], 
                    'II_USA FEDATLANTA 4'  : ['EDI', 'Edge Corporation - Investment'], 
                    'II_USA FEDATLANTA 5'  : ['FHD', 'Financial Holding Company / BHC (Note: Can be a domestic or foreign domiciled holding company)'],
                    'II_USA FEDATLANTA 6'  : ['SLHC', 'Savings and Loan Holding Company']
                    },
'II_USA FEDBOS':{'II_USA FEDBOS 1' : ['SMB', 'State Member Bank'],
                    'II_USA FEDBOS 2'  : ['BHC', 'Bank Holding Company'],
                    'II_USA FEDBOS 3'  : ['EDI', 'Edge Corporation - Investment'],
                    'II_USA FEDBOS 4'  : ['FHD', 'Financial Holding Company / BHC (Note: Can be a domestic or foreign domiciled holding company)'],
                    'II_USA FEDBOS 5'  : ['IHC', 'Intermediate Holding Company'],
                    'II_USA FEDBOS 6'  : ['SLHC', 'Savings and Loan Holding Company'],

                    },
'II_USA FEDCHICAGO':{'II_USA FEDCHICAGO 1' : ['SMB', 'State Member Bank'],
                    'II_USA FEDCHICAGO 2'  : ['BHC', 'Bank Holding Company'],
                    'II_USA FEDCHICAGO 3'  : ['EDI', 'Edge Corporation - Investment'],
                    'II_USA FEDCHICAGO 4'  : ['FHD', 'Financial Holding Company / BHC (Note: Can be a domestic or foreign domiciled holding company)'],
                    'II_USA FEDCHICAGO 5'  : ['IHC', 'Intermediate Holding Company'],
                    'II_USA FEDCHICAGO 6'  : ['SLHC', 'Savings and Loan Holding Company']

                    },
'II_USA FEDCLE'   :{'II_USA FEDCLE 3'  : ['SMB', 'State Member Bank'],
                    'II_USA FEDCLE 4'  : ['AGI', 'Agreement Corporation - Investment'],
                    'II_USA FEDCLE 5'  : ['BHC', 'Bank Holding Company'],
                    'II_USA FEDCLE 6'  : ['FHD', 'Financial Holding Company / BHC (Note: Can be a domestic or foreign domiciled holding company)'],
                    'II_USA FEDCLE 7'  : ['SLHC', 'Savings and Loan Holding Company']
                    },
'II_USA FEDDALLAS':{'II_USA FEDDALLAS 1' : ['SMB', 'State Member Bank'],
                    'II_USA FEDDALLAS 2' : ['BHC', 'Bank Holding Company'],
                    'II_USA FEDDALLAS 3' : ['FHD', 'Financial Holding Company / BHC (Note: Can be a domestic or foreign domiciled holding company)'],
                    'II_USA FEDDALLAS 4' : ['SLHC', 'Savings and Loan Holding Company'],
                    },
'II_USA FEDKANSAS':{'II_USA FEDKANSAS 1' : ['SMB', 'State Member Bank'],
                    'II_USA FEDKANSAS 2'  : ['AGI', 'Agreement Corporation - Investment'],
                    'II_USA FEDKANSAS 3'  : ['BHC', 'Bank Holding Company'],
                    'II_USA FEDKANSAS 4'  : ['EDI', 'Edge Corporation - Investment'],
                    'II_USA FEDKANSAS 5'  : ['FHD', 'Financial Holding Company / BHC (Note: Can be a domestic or foreign domiciled holding company)'],
                    'II_USA FEDKANSAS 6'  : ['SLHC', 'Savings and Loan Holding Company'],
                    },
'II_USA FEDMINNEAPOLIS':{'II_USA FEDMINNEAPOLIS 1' : ['SMB', 'State Member Bank'],
                    'II_USA FEDMINNEAPOLIS 2'  : ['AGI', 'Agreement Corporation - Investment'],
                    'II_USA FEDMINNEAPOLIS 3'  : ['BHC', 'Bank Holding Company'],
                    'II_USA FEDMINNEAPOLIS 4'  : ['FHD', 'Financial Holding Company / BHC (Note: Can be a domestic or foreign domiciled holding company)'],
                    'II_USA FEDMINNEAPOLIS 5'  : ['SLHC', 'Savings and Loan Holding Company'],
                    },
'II_USA FEDNY':{'II_USA FEDNY 1'  : ['SMB', 'State Member Bank'],
                'II_USA FEDNY 2'  : ['AGI', 'Agreement Corporation - Investment'], 
                'II_USA FEDNY 3'  : ['BHC', 'Bank Holding Company'], 
                'II_USA FEDNY 4'  : ['EDB', 'Edge Corporation - Banking'],
                'II_USA FEDNY 5'  : ['EDI', 'Edge Corporation - Investment'], 
                'II_USA FEDNY 6'  : ['FHD', 'Financial Holding Company / BHC (Note: Can be a domestic or foreign domiciled holding company)'],
                'II_USA FEDNY 7'  : ['FHF', 'Financial Holding Company / FBO'],
                'II_USA FEDNY 8'  : ['IHC', 'Intermediate Holding Company'],
                'II_USA FEDNY 9'  : ['MTC', 'Non-deposit Trust Company - Member'],
                'II_USA FEDNY 10' : ['SLHC', 'Savings and Loan Holding Company'],
                    },

'II_USA FEDRICH':{'II_USA FEDRICH 1': ['BHC', 'Bank Holding Company'], 
                'II_USA FEDRICH 2'  : ['SMB', 'State Member Bank'],
                'II_USA FEDRICH 3'  : ['SLHC', 'Savings and Loan Holding Company'],
                'II_USA FEDRICH 4'  : ['AGI', 'Agreement Corporation - Investment'],  
                'II_USA FEDRICH 5'  : ['EDI', 'Edge Corporation - Investment'],
                'II_USA FEDRICH 6'  : ['FHD', 'Financial Holding Company / BHC (Note: Can be a domestic or foreign domiciled holding company)'], 
                },
'II_USA FEDSTLOUIS':{'II_USA FEDSTLOUIS 1': ['SMB', 'State Member Bank'],
                'II_USA FEDSTLOUIS 2'  : ['BHC', 'Bank Holding Company'], 
                'II_USA FEDSTLOUIS 3'  : ['FHD', 'Financial Holding Company / BHC (Note: Can be a domestic or foreign domiciled holding company)'], 
                'II_USA FEDSTLOUIS 4'  : ['SLHC', 'Savings and Loan Holding Company'],
                },

'II_USA FEDSF':{'II_USA FEDSF 1': ['SMB', 'State Member Bank'],
                'II_USA FEDSF 4'  : ['EDI', 'Edge Corporation - Investment'], 
                'II_USA FEDSF 6'  : ['SLHC', 'Savings and Loan Holding Company'], 
                'II_USA FEDSF 7'  : ['BHC', 'Bank Holding Company'],
                'II_USA FEDSF 9'  : ['FHD', 'Financial Holding Company / BHC (Note: Can be a domestic or foreign domiciled holding company)'],
                }
}

DIST_FRS = {
    'II_USA FEDBOS'         : 1,
    'II_USA FEDNY'          : 2,
    'II_USA FEDPHIL'        : 3,
    'II_USA FEDCLE'         : 4,
    'II_USA FEDRICH'        : 5,
    'II_USA FEDATLANTA'     : 6,
    'II_USA FEDCHICAGO'     : 7,
    'II_USA FEDSTLOUIS'     : 8,
    'II_USA FEDMINNEAPOLIS' : 9,
    'II_USA FEDKANSAS'      : 10,
    'II_USA FEDDALLAS'      : 11,
    'II_USA FEDSF'     : 12,
    # 'II_USA FEDSANFRANCISCO': 13, 
}

now=datetime.datetime.now()
processdate = now.strftime('%Y-%m-%d')
scriptfolder=os.getcwd()
os.chdir(scriptfolder)

date_now = str(now).replace(":",".")[:-7]

tempfolder=os.path.join(scriptfolder, 'tempfolder') #if files are downloaded during the process
if os.path.exists(tempfolder):
    for rem in os.listdir(tempfolder):
        os.remove(os.path.join(tempfolder, rem))
else:
    os.mkdir(tempfolder)
#------------------------------------------------ Begin_Fouction ----------------------------------------
def unzip(source_path, output_path):
    with ZipFile(source_path, 'r') as zip_:
        zip_.extractall(output_path)
        
def bourange_same_length_array(sqldict) :
    maxlen = len(sqldict['ListProcessDate'])
    for key, val in sqldict.items():
        if len(sqldict[key]) != maxlen:
            empty = []
            total_empty = maxlen - len(sqldict[key])
            for i in range(total_empty):
                empty.append('')
            sqldict[key]=sqldict[key]+empty
    return sqldict    
#------------------------------------------------ Begin_Main ---------------------------------------- 
file = 'CSV_ATTRIBUTES_ACTIVE.zip'
filePath = os.path.join(scriptfolder, file)
unzip(filePath, tempfolder)
file = [ele for ele in os.listdir(tempfolder) if ele.endswith('.CSV')][0]
filePath = os.path.join(tempfolder, file)
df = pd.read_csv(filePath)
df = df.fillna("")  # subsitute nan with empty strings
df.replace(0, '', inplace=True)
df.replace('0', '', inplace=True)
print(f'[INFO] : Import {file} | {len(df)} row')

for j, regdict in enumerate(regulators):
    print(f'[INFO] : * Regulator processing {j+1}/{len(regulators[regdict])} | {len(regulators)} data | regulator Name : {regdict}')

    sqldict={'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 
		'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [], 
		'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [], 
		'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],
		'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [], 
		'Phone - Mother company': [], 'Check': []}

    for k, reg in enumerate(regulators[regdict]):
        # df_reg=df[(df['DIST_FRS']==3) & (df['PRIM_FED_REG']=='FRS') & (df['ENTITY_TYPE']==regdict[reg])]
        df_reg=df[(df['DIST_FRS']==DIST_FRS[regdict]) & (df['ENTITY_TYPE']==regulators[regdict][reg][0])]
        print(f'[INFO] : -- Reg list processing {k+1}/{len(regulators[regdict])} | {len(df_reg)} data | reg list : {reg}')
        
        for index, row in df_reg.iterrows():
            sqldict['Name'].append(row['NM_LGL'])
            sqldict['Address_1'].append(row['STREET_LINE1'])
            sqldict['Address_2'].append(row['STATE_ABBR_NM'])
            sqldict['Zip'].append(row['ZIP_CD'])
            sqldict['City'].append(row['CITY'])
            sqldict['Typology'].append(row['ENTITY_TYPE'])
            sqldict['RegulationType'].append('Supervised')
            sqldict['InternalID_1_type'].append('ID RSSD')
            sqldict['InternalID_1'].append(row['#ID_RSSD'])
            sqldict['InternalID_2'].append(row['ID_ABA_PRIM']) 
            sqldict["InternalID_2_type"].append("ABA Number") 
            # sqldict['InternalID_3'].append(row['ID_THRIFT'])
            # sqldict["InternalID_3_type"].append("ID_THRIFT")  
            sqldict['LEI Code'].append(row['ID_LEI'])
            sqldict['Website'].append(row['URL'])
            sqldict['Cntry'].append(row['CNTRY_NM'])

            sqldict['ListProcessDate'].append(processdate)
            sqldict['RegCtry'].append(reg.split(' ')[0])
            sqldict['RegCode'].append(reg.split(' ')[1])
            sqldict['ListCode'].append(reg.split(' ')[-1])
            sqldict['ListName'].append(regulators[regdict][reg][1])
                
        sqldict = bourange_same_length_array(sqldict)

        if export_pdf :
            def export_list_to_pdf(data_list, pdf_filename):
                c = canvas.Canvas(pdf_filename, pagesize=letter)
                c.setFont("Helvetica", 12)
                x = 72
                y = 720
                max_lines_per_page = 33 # Maximum number of lines per page
                line_count = 0
                
                # Add each item from the list to the PDF
                for item in data_list:
                    c.drawString(x, y, str(item))
                    y -= 20  # Move to the next line
                    line_count += 1
                    if line_count >= max_lines_per_page: # Check if we need to add a new page
                        c.showPage()  # Add a new page
                        c.setFont("Helvetica", 12)  # Reset font
                        x = 72  # Reset x position
                        y = 720  # Reset y position
                        line_count = 0  # Reset line counter
                        
                c.save()

            export_list_to_pdf(list(df_reg['NM_LGL']), os.path.join(tempfolder, f"{regdict} data {date_now} - {reg.split(' ')[-1]}.pdf"))


    filename = '{} data {}.xlsx'.format(regdict, date_now)
    writer = ExcelWriter(os.path.join(tempfolder, filename), engine='openpyxl')
    sleep(1)
    df_save=pd.DataFrame(sqldict)
    df_save.to_excel(writer, 'SQL Ready', index=False)
    writer.save()
    writer.close()
    sleep(1)

# os.remove(filePath)
print('[INFO] : Finish | OK')


C:\Users\siewekoa\AppData\Local\Temp\ipykernel_26032\2884247225.py:155: DtypeWarning: Columns (35,45,47,63) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(filePath)


[INFO] : Import CSV_ATTRIBUTES_ACTIVE.CSV | 61177 row
[INFO] : * Regulator processing 1/7 | 12 data | regulator Name : II_USA FEDPHIL
[INFO] : -- Reg list processing 1/7 | 17 data | reg list : II_USA FEDPHIL 1
[INFO] : -- Reg list processing 2/7 | 2 data | reg list : II_USA FEDPHIL 2
[INFO] : -- Reg list processing 3/7 | 77 data | reg list : II_USA FEDPHIL 3
[INFO] : -- Reg list processing 4/7 | 3 data | reg list : II_USA FEDPHIL 4
[INFO] : -- Reg list processing 5/7 | 19 data | reg list : II_USA FEDPHIL 5
[INFO] : -- Reg list processing 6/7 | 1 data | reg list : II_USA FEDPHIL 6
[INFO] : -- Reg list processing 7/7 | 8 data | reg list : II_USA FEDPHIL 7
[INFO] : * Regulator processing 2/6 | 12 data | regulator Name : II_USA FEDATLANTA
[INFO] : -- Reg list processing 1/6 | 33 data | reg list : II_USA FEDATLANTA 1
[INFO] : -- Reg list processing 2/6 | 1 data | reg list : II_USA FEDATLANTA 2
[INFO] : -- Reg list processing 3/6 | 359 data | reg list : II_USA FEDATLANTA 3
[INFO] : -- Reg li